# RAG Evaluation with DeepEval

This notebook evaluates a RAG (Retrieval-Augmented Generation) pipeline using the [DeepEval](https://github.com/confident-ai/deepeval) framework.

It uses the file `evaluation_set.json`, which contains for each question:
- `Question`: the user query
- `output_answer`: the answer generated by the RAG pipeline
- `output_chunks`: the chunks generated by the RAG pipeline
- `expected_answer`: ground truth
- `expected_chunks`: expected chunks

We evaluate two things:

1. **Retrieval quality** (how good are the retrieved chunks?)
   - `ContextualPrecisionMetric` - are the most relevant chunks ranked higher?
   - `ContextualRecallMetric` - do the retrieved chunks contain everything needed to produce the expected answer?
   - `ContextualRelevancyMetric` - how relevant, on average, are the retrieved chunks to the question?

2. **Answer quality** (how good is the generated answer?)
   - `FaithfulnessMetric` - is the predicted answer factually consistent with the retrieved context (i.e. not hallucinating)?
   - `AnswerRelevancyMetric` - does the predicted answer actually address the question?
   - `GEval` (correctness) - how close is the predicted answer to the expected ground-truth answer?

BLEU=BP⋅exp(∑n=1,N w_n * log p_n)

Brevity Penalty ( BP ): BP=1 if candidate length c> reference length r ; otherwise BP=e^1−r/c.

N-gram Precision (p_n): Ratio of matching n-grams in the candidate to total n-grams in the candidate, with counts clipped by reference counts.

Weights (w_n): Typically uniform (e.g., 1/N ).

ROUGE (Recall-Oriented Understudy for Gisting Evaluation) measures recall based on overlapping n-grams, longest common subsequences (LCS), or skip bigrams.

The most common variant, ROUGE-N, 

calculates recall as:
ROUGE-N= ∑n-gram∈reference min(Count_candidate, Count_reference) / ∑n-gram∈reference Count(n-gram)


ROUGE-L: Uses the Longest Common Subsequence (LCS) to evaluate sentence-level similarity, often combining precision and recall into an F1 score. 

ROUGE-S: Measures skip-bigram overlap to capture flexible word ordering.

All metrics in DeepEval are backed by an LLM-as-judge. This notebook uses **open-source models hosted on [Groq](https://console.groq.com)** as the judge (via Groq's OpenAI-compatible endpoint), so you'll need a `GROQ_API_KEY` set up before running the evaluation cells.

## 1. Setup

In [ ]:
import os

# We use Groq-hosted open-source models as the LLM judge, via DeepEval's
# generic OpenAI-compatible LocalModel wrapper pointed at Groq's endpoint.
# Set your Groq API key here, or export it as an environment variable before starting Jupyter.
# os.environ["GROQ_API_KEY"] = "gsk_..."

GROQ_BASE_URL = "https://api.groq.com/openai/v1"

assert os.environ.get("GROQ_API_KEY"), (
    "Please set GROQ_API_KEY (from https://console.groq.com/keys) before running evaluations."
)

In [ ]:
import json
import pandas as pd
import json
import os 
from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.metrics import (
    ContextualPrecisionMetric,
    ContextualRecallMetric,
    ContextualRelevancyMetric,
    FaithfulnessMetric,
    AnswerRelevancyMetric,
    GEval,
)
from deepeval.test_case import LLMTestCaseParams
from deepeval.models import LocalModel  # generic OpenAI-compatible wrapper, used here for Groq

## 2. Load the data

Update `DATA_PATH` if you place the file elsewhere.

In [ ]:
# evaluation_set.json is expected in the same directory as this notebook.
DATA_PATH = "../../eval_dataset/evaluation_set.json" # Change path as per your evaluation_set.json

with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

print(f"Loaded {len(raw_data)} evaluation records")
print("Keys:", raw_data[0].keys())


In [ ]:
# Quick look at one record
sample = raw_data[0]

print("Question:", sample["question"])
print("\nExpected Answer:", sample["expected_answer"])
print("\nOutput Answer:", sample["output_answer"][:500], "..." if len(sample["output_answer"]) > 500 else "")
print("\nNumber of expected chunks:", len(sample.get("expected_chunks", [])))
print("Number of output chunks:", len(sample.get("output_chunks", [])))

if sample.get("expected_chunks"):
    print("\nExpected chunk preview:", sample["expected_chunks"][0].get("text", "")[:300], "...")

if sample.get("output_chunks"):
    print("\nOutput chunk preview:", sample["output_chunks"][0].get("text", "")[:300], "...")


## 3. Build DeepEval `LLMTestCase`s

Each record in `evaluation_set.json` has this structure:

```text
question
document_id
expected_chunks
expected_context
expected_answer
output_chunks
output_answer
```

Mapping used here:
- `input` -> `question`
- `actual_output` -> `output_answer`
- `expected_output` -> `expected_answer`
- `retrieval_context` -> text from `output_chunks` (what the retriever actually returned)
- `context` -> text from `expected_chunks` / `expected_context` (the expected/gold context)


In [ ]:
def chunk_text(chunk):
    """Extract chunk text from the evaluation-set chunk format."""
    if isinstance(chunk, str):
        return chunk
    if isinstance(chunk, dict):
        return chunk.get("text", "")
    return str(chunk)


def build_test_case(record: dict) -> LLMTestCase:
    # Actual retrieved context: what the RAG retriever returned.
    output_chunks = record.get("output_chunks", [])
    retrieval_context = [chunk_text(chunk) for chunk in output_chunks]

    # Gold/expected context. Prefer expected_chunks when available because
    # it preserves the individual expected chunks used for retrieval evaluation.
    expected_chunks = record.get("expected_chunks", [])
    expected_context = record.get("expected_context", [])
    
    if expected_chunks:
        gold_context = [chunk_text(chunk) for chunk in expected_chunks]
    elif isinstance(expected_context, list):
        gold_context = [chunk_text(chunk) for chunk in expected_context]
    elif expected_context:
        gold_context = [str(expected_context)]
    else:
        gold_context = []

    return LLMTestCase(
        input=record["question"],
        actual_output=record.get("output_answer", ""),
        expected_output=record.get("expected_answer", ""),
        retrieval_context=retrieval_context,
        context=gold_context,
    )


test_cases = [build_test_case(r) for r in raw_data]
print(f"Built {len(test_cases)} test cases")


## 4. Define metrics

### Retrieval metrics
- **ContextualPrecision**: evaluates whether the retrieved `output_chunks` are relevant and appropriately ranked relative to the expected context.
- **ContextualRecall**: evaluates how well the retrieved `output_chunks` cover the expected information/context.
- **ContextualRelevancy**: evaluates whether the retrieved chunks are relevant to the question.

### Answer metrics
- **Faithfulness**: evaluates whether `output_answer` is supported by the retrieved `output_chunks`.
- **AnswerRelevancy**: evaluates whether `output_answer` addresses the question.
- **Correctness (GEval)**: compares `output_answer` with `expected_answer`.


### Judge model: open-source models on Groq

Groq exposes an OpenAI-compatible endpoint (`https://api.groq.com/openai/v1`), so we point DeepEval's generic `LocalModel` wrapper at it instead of using OpenAI directly. This lets every metric run its judging with an open-weight model served by Groq.

Current Groq **production** text models (as of Aug 2026), excluding `qwen/qwen3.6-27b` per requirement:

| Model ID | Notes |
|---|---|
| `openai/gpt-oss-120b` | Best overall quality/reasoning of the open text models on Groq - recommended default judge |
| `llama-3.3-70b-versatile` | Strong general-purpose alternative |
| `openai/gpt-oss-20b` | Smaller/faster, cheaper, slightly less reliable judgments |
| `llama-3.1-8b-instant` | Fastest/cheapest, best for quick smoke tests rather than final scoring |

Pick one via `JUDGE_MODEL` below. You can also use two different models - e.g. a strong one for correctness/faithfulness and a fast one for cheaper metrics - by creating separate `LocalModel` instances.

In [ ]:
THRESHOLD = 0.5
JUDGE_MODEL = "openai/gpt-oss-20b" # "openai/gpt-oss-120b" 

judge = LocalModel(
    model=JUDGE_MODEL,
    api_key=os.environ["GROQ_API_KEY"],
    base_url=GROQ_BASE_URL,
    temperature=0,
)

# --- Retrieval-quality metrics ---
contextual_precision = ContextualPrecisionMetric(threshold=THRESHOLD, model=judge, include_reason=True)
contextual_recall = ContextualRecallMetric(threshold=THRESHOLD, model=judge, include_reason=True)
contextual_relevancy = ContextualRelevancyMetric(threshold=THRESHOLD, model=judge, include_reason=True)

# --- Answer-quality metrics ---
faithfulness = FaithfulnessMetric(threshold=THRESHOLD, model=judge, include_reason=True)
answer_relevancy = AnswerRelevancyMetric(threshold=THRESHOLD, model=judge, include_reason=True)

correctness = GEval(
    name="Correctness",
    criteria=(
        "Determine whether the actual output is factually correct and semantically "
        "equivalent to the expected output, given the input question. Minor differences "
        "in wording, structure, or level of detail are fine as long as the core facts match."
    ),
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT,
        LLMTestCaseParams.EXPECTED_OUTPUT,
    ],
    threshold=THRESHOLD,
    model=judge,
)

retrieval_metrics = [contextual_precision, contextual_recall, contextual_relevancy]  # Add contextual_recall/contextual_relevancy when needed
answer_metrics = [faithfulness]  # Add answer_relevancy/correctness when needed
all_metrics = retrieval_metrics + answer_metrics


## 5. Run the evaluation

`deepeval.evaluate` runs every metric against every test case and returns structured results.
This will make LLM calls for each (test case, metric) pair, so with 10 questions x 6 metrics expect ~60+ judge calls (some metrics like `ContextualPrecision`/`Recall`/`Relevancy` and `Faithfulness` make several sub-calls per test case to score individual chunks/claims, so actual call volume will be higher).

Groq's developer-tier rate limits are generous but finite (see the table above, e.g. 1K RPM for `gpt-oss-120b`) - if you hit 429s on a larger dataset, either switch `JUDGE_MODEL` to a higher-limit model, add `async_config`/concurrency limits, or run in smaller batches.

In [ ]:
eval_results = evaluate(
    test_cases=list(test_cases),
    metrics=all_metrics,
)


## 6. Collect results into a DataFrame

Flatten the per-test-case, per-metric results into a tidy table for analysis/export.

In [ ]:
rows = []
for i, tc_result in enumerate(eval_results.test_results):
    original = raw_data[i]
    row = {
        "question": tc_result.input,
        "document_id": original.get("document_id"),
        "expected_answer": tc_result.expected_output,
        "output_answer": tc_result.actual_output,
        "expected_chunk_count": len(original.get("expected_chunks", [])),
        "output_chunk_count": len(original.get("output_chunks", [])),
    }
    for metric_data in tc_result.metrics_data:
        row[f"{metric_data.name}_score"] = metric_data.score
        row[f"{metric_data.name}_success"] = metric_data.success
        row[f"{metric_data.name}_reason"] = metric_data.reason
    rows.append(row)

results_df = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 120)
results_df.head()


## 7. Summary scores

Average score per metric across all questions, plus pass rate (fraction of questions where score >= threshold).

In [ ]:
score_cols = [c for c in results_df.columns if c.endswith("_score")]
success_cols = [c for c in results_df.columns if c.endswith("_success")]

summary = pd.DataFrame({
    "metric": [c.replace("_score", "") for c in score_cols],
    "avg_score": [results_df[c].mean() for c in score_cols],
    "pass_rate": [results_df[sc].mean() for sc in success_cols],
})
summary

In [ ]:
print("=== Retrieval metrics (chunk quality) ===")
display(summary[summary["metric"].str.contains("Contextual")])

print("\n=== Answer metrics (generation quality) ===")
display(summary[~summary["metric"].str.contains("Contextual")])

## 8. Inspect low-scoring examples

Useful for debugging: find questions where retrieval or generation quality was weak.

In [ ]:
def show_low_scores(df: pd.DataFrame, metric: str, n: int = 3):
    col = f"{metric}_score"
    reason_col = f"{metric}_reason"
    worst = df.sort_values(col).head(n)
    for _, r in worst.iterrows():
        print(f"Question: {r['question']}")
        print(f"{metric} score: {r[col]:.2f}")
        print(f"Reason: {r[reason_col]}")
        print("-" * 80)

# Example: worst 3 by Faithfulness
show_low_scores(results_df, "Faithfulness", n=3)


In [ ]:
# Example: worst 3 by Contextual Recall
show_low_scores(results_df, "Contextual Recall", n=3)

## 9. Export results

In [ ]:
results_df.to_csv("deepeval_results_detailed.csv", index=False)
summary.to_csv("deepeval_results_summary.csv", index=False)

# Also save the evaluation configuration for reproducibility.
evaluation_config = {
    "data_file": "evaluation_set.json",
    "judge_model": JUDGE_MODEL,
    "threshold": THRESHOLD,
    "retrieval_metrics": [m.name for m in retrieval_metrics],
    "answer_metrics": [m.name for m in answer_metrics],
}
with open("deepeval_evaluation_config.json", "w", encoding="utf-8") as f:
    json.dump(evaluation_config, f, indent=2)

print("Saved: deepeval_results_detailed.csv, deepeval_results_summary.csv, deepeval_evaluation_config.json")


## Notes / customization tips

- `evaluation_set.json` must be in the same directory as this notebook.
- `output_chunks` are treated as the actual retrieved context.
- `expected_chunks` are treated as the gold/expected context.
- `output_answer` is the answer generated by your RAG pipeline.
- `expected_answer` is the reference answer.
- `Faithfulness` checks the generated answer against the actual retrieved `output_chunks`.
- Retrieval metrics compare the actual retrieved context with the expected/gold context.
- Start with one metric while debugging, then enable the others as needed.
- `openai/gpt-oss-120b` is used as the judge here; `openai/gpt-oss-20b` is a lower-cost alternative.
